# e5_64x64_8_layers_no_aug_SGD_weight_decay

This is the baseline experiment. It includes following components in the model:
* 64x64x3 input size
* Data Augmentations
* 8 Convolutional Layers
* Pooling Layer after skipping 1 convolutional Layer each
* SGD optimizer

### Importing required modules
Importing basic modules for training:
* torch
* datasets and transform
* dataloader

Importing Architecture:
* Architecture

Importing custom utils
* Trainer
* tester
* gen_line_charts

In [ ]:
# torch and modules
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Main architecture
from Architecture import Architecture

# custom utils and modules
from modules.trainer import Trainer
from modules.tester import test
from modules.charts import gen_line_charts

## Load data
Use datasets, transforms and DataLoader to:
* LoadData
* Resize image to 64x64
* Randomly flip images horizontally
* Randomly rotate image by +-10 deg
* Add Color Jitters
* Randomly move the image
* transform to tensor
* Add a small Random Erasing between 2% to 10% erasing

For all three datasets: train, val and test

In [ ]:
image_size = 64

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor()
])

# make train sets
train_datasets = datasets.ImageFolder("./datasets/train", transform)
train_loader = DataLoader(
    dataset=train_datasets, 
    batch_size=32,
    num_workers=4,
    pin_memory=True,
    shuffle=True,
    persistent_workers=True
)

# make validation sets
val_datasets = datasets.ImageFolder("./datasets/val", transform)
val_loader = DataLoader(
    dataset=val_datasets, 
    batch_size=32,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

# make test sets
test_datasets = datasets.ImageFolder("./datasets/test", transform)
test_loader = DataLoader(
    dataset=test_datasets, 
    batch_size=32,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

## Define and Use Architecture
define a model by calling Architecture class and add layers to it:
* 8 Convolutional Layer each followed with skipping pooling layer
* 1 Flatten Layer followed with a 128 fully connected neural network
* 1 Linear layer to generate 3 class logits

In [ ]:
model = Architecture()

Using function to loop adding 8 convolutional layers with a skip pooling layer 

In [ ]:
def add_conv_layers(layers=1, skip_pool=0):
    # define in and out channels
    in_channels = 3
    out_channels = 8
    # size
    size = image_size
    # skip pool increase by one for calculation
    skip_pool = skip_pool+1
    # save a trainable parameters
    total_conv_params = 0
    # loop each layers to add on model
    for layer in range(layers):
        # Convolutional Layer
        model.add(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )
        # calculate trainable params
        total_conv_params += (((3*3*in_channels) + 1) * out_channels) + (2*out_channels)
        # Pooling Layer
        if  (layer + 1) % skip_pool:
            model.add(nn.MaxPool2d(2,2))
            size = size//2
        # update in and out channels
        if (layer<layers-1):
            in_channels = out_channels
            out_channels = out_channels*2
    # return total convolutional layer parameters
    return total_conv_params, out_channels, size

Call the `add_conv_layers()` and store parameters

In [ ]:
conv_params, out_channels, size = add_conv_layers(8, 1)

# print conv params and out_channels
print(f"Total Trainable Parameters: {conv_params}")
print(f"Final Features: {out_channels}")
print(f"Final Feature size: {size}x{size}")

use `model.add()` to add:
* Flatten Layer (64x4x4)
* Hidden Layer (64x4x4->128)
* Output Layer (128->3)

In [ ]:
# calculate input
n_in = out_channels*size*size
print(f"Input vector size = {n_in}")

model.add(
    # Flatten
    nn.Flatten(),
    # Hidden Layer
    nn.Linear(n_in, 128),
    nn.ReLU(),
    # Output Layer
    nn.Linear(128, 3),
    nn.ReLU()
)

Calculating total parameters of the model

In [ ]:
total_trainable_parameters = ((n_in*128)+128) + ((128*3)+3)
print(f"Total trainable parameters = {total_trainable_parameters}")

### Defining optimizer and Criterion
Optimizer:
* SGD
* learning rate = 2e-3
* no weight decay

Critetion:
* nn.CrossEntropy()

In [ ]:
# optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=2e-3, weight_decay=1e-5)
# criterion 
criterion = nn.CrossEntropyLoss()

## Train the model
Use Trainer Function to train the complete model for 100 epochs

In [ ]:
trainer = Trainer(
    model,
    train_loader,
    val_loader,
    optimizer,
    "cuda",
    criterion
)

In [ ]:
save_path = "./experiments/e5_64x64_8_layers_no_aug_SGD_weight_decay"

In [ ]:
metrics = trainer.fit(100, save_path, 1)

## Plot the training metrics
Plotting line chart of training metrics that shows curves of loss, accuracy, precision, recall and f1-score for training set and validation set. 

In [ ]:
gen_line_charts(metrics, save_path, "training_metrics_graph.png", ["train_", "val_"])

### Model Evaluation
The loss curve on training set tends to decrease approaching 0. The training loss also decreased a little till approximately 0.67. The model is learning quiickly on the training set. By the epoch 30 training loss have reached below 0.02 while the validation loss was still around 0.64. Fluctuations with extreme peaks is visible on the validation loss reaching approximately 0.7. Indicating the model is learning on training data but cannot generalize for validation throughout the validation set. The loss curves shows similarity as the loss curves in the first experiment but on closer look the decrease on training loss is comparatively more smooth and the validation loss had more than one peaks approximating to 0.7 on the first experiment. 

The accuracy, precision, recall and F1-score curve shows a visible gap between training set and validation set. The training curve reaches 1.00 flat at around 30 epochs while the validation curve is at approximately 0.8 mostly throughout the 100 epochs. The validation curves shows extreme fluctuations suddenly dropping below 0.4. These curves looks similar to the first experiment but the curves of training set on current experiment seems much smoother while approaching 1. Also the curves on validation sets are slightly curvy while it approaches to 0.8.

Overall the model is performing similar to first experiment. The removal of augmentation did improve the convergence speed. The visible gap between the training and validation curves suggest overfitting. A possible reason of the fluctuations can be the uncontrolled constant learning rate, so it could be further studied by adding weight decay of 1e-5.